# Notebook 06 — Stage 2: replicate harmonisation and duplicate checks

**Role in the pipeline:** the first stage that operates *across* rows
rather than within them. Reads Stage 1B's `analysis_ready_samples.csv`,
adds two grouping keys (`sample_month`, `depth_round_m`), checks for
duplicates, and computes per-replicate-group means + standard
deviations. Flags rows whose group has a metadata / provenance / QC /
source conflict or where the SD of a metric exceeds the GOA-ON
"weather" precision threshold.

```text
05_stage1b.ipynb
   └── <stage1b_out>/data/analysis_ready_samples.csv
                                │
                                ▼
                  06_stage2.ipynb            ◄── THIS NOTEBOOK
                     • re-resolve canonical aliases (defensive)
                     • add sample_month, depth_round_m
                     • duplicate detection (per-row + per-group summary)
                     • replicate harmonisation (means + SDs)
                     • conflict + SD-threshold flags
                     • write enhanced.csv (rows + flags) +
                       per-group tables (replicate means, conflicts, ...)
                                                  │
                                                  ▼
                                       07_stage3.ipynb → 08_stage4
```

**What Stage 2 does NOT do.** It does not rebuild `ph_best`,
`ph_co2sys`, `ta_best_umolkg`, `pco2_best_uatm`, or `dic_best_umol_kg`.
Those are Stage 1B's job and are preserved unchanged. Stage 2 only
*aggregates* and *flags*; it does not silently overwrite values.


## Parameters

Single tagged `parameters` cell. Default `INPUT_CSV` points at the new
short Stage 1B path (`<stage1b_out>/data/analysis_ready_samples.csv`),
not the original's flat `oa_stage1b_outputs\analysis_ready_samples.csv`
(which never existed in the refactored layout).


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# These are intentionally None so the notebook does not accidentally run on
# a personal local path. run_pipeline.sh supplies both values with Papermill.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep-merges onto roap.stage2.STAGE2_DEFAULTS.
CONFIG_PATH = None

# --- Stage 2 behaviour ---------------------------------------------
# Rounding precision for the replicate grouping key.
DEPTH_ROUND_DECIMALS = 1

# If True, skip Parquet writes (CSV only). Parquet failures are tolerated
# and recorded in the manifest.
NO_PARQUET = False

# Useful during development: prepare everything, write nothing.
DRY_RUN = False

# Optional debugging aid.
PRINT_COLUMNS = False


## Setup

All Stage 2 logic lives in `roap.stage2`: `run(df, stage_config)` re-resolves
canonical aliases, adds grouping helpers, checks duplicates and harmonises
replicates, and returns a `StageResult` without touching disk. This notebook
reads the input, calls `run`, shows the tables and writes the result.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from roap import stage2
from roap.common import as_bool, die, is_none_like
from roap.stage_config import load_stage_config
from roap.stage_result import write_stage_result

## Load input and canonicalise

The first step *re-runs* canonical alias resolution. Why, when Stage 1B
should already have done it? Because Stage 2 should also work when fed
some other CSV that happens to have the right shape but uses different
column names. If you feed it a clean Stage 1B output the alias step is
a no-op; if you feed it something exotic it gives you the chance to
recover.


In [ ]:
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)
PRINT_COLUMNS = as_bool(PRINT_COLUMNS)

try:
    DEPTH_ROUND_DECIMALS = int(DEPTH_ROUND_DECIMALS)
except Exception:
    die(f"DEPTH_ROUND_DECIMALS must be an integer, got {DEPTH_ROUND_DECIMALS!r}")

if is_none_like(INPUT_CSV):
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")
if is_none_like(OUT_DIR):
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")
config_path = None if is_none_like(CONFIG_PATH) else str(CONFIG_PATH).strip()

input_csv = Path(str(INPUT_CSV).strip()).expanduser().resolve()
if not input_csv.exists():
    die(f"File not found: {input_csv}\nDid Stage 1B run successfully? Stage 2 reads analysis_ready_samples.csv.")
if input_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {input_csv.name}")
out_root = Path(str(OUT_DIR).strip()).expanduser().resolve()

# Stage 2 defaults <- this Stage's optional config file, and nothing else (#41).
stage_cfg = load_stage_config("stage2", config_path)
for _warning in stage_cfg.warnings:
    print(f"WARNING: {_warning}")

try:
    result = stage2.run(
        pd.read_csv(input_csv), stage_cfg, source=str(input_csv), depth_round_decimals=DEPTH_ROUND_DECIMALS
    )
except ValueError as exc:
    die(str(exc))

result.manifest["output_root"] = str(out_root)
result.manifest["parameters"].update(
    {
        "INPUT_CSV": str(input_csv),
        "OUT_DIR": str(out_root),
        "CONFIG_PATH": CONFIG_PATH,
        "config_path_resolved": config_path,
        "NO_PARQUET": NO_PARQUET,
        "DRY_RUN": DRY_RUN,
    }
)
df = result.data["data/enhanced.csv"]
tables = result.tables
counts = result.manifest["row_counts"]

if PRINT_COLUMNS:
    print("\nColumns:")
    for col in df.columns:
        print(f"  {col}")
print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {df.shape[1]}")
print(f"Output root: {out_root}")

## Canonical-field presence inventory

A one-row-per-promised-column view: required vs expected, present
vs missing, count and percent of non-NA values. Use this to verify
Stage 1B really wrote what Stage 2 needs.


In [ ]:
if display is not None:
    display(tables["tables/canonical_presence.csv"].head(200))
else:
    print(tables["tables/canonical_presence.csv"].head(200).to_string(index=False))

## Add grouping helpers

Two new columns get added:

- `sample_month` — monthly period string derived from `sample_date`
  (e.g. `"2024-03"`). Used by the replicate consistency-check, not
  by the replicate-group key (the group key uses `sample_date` itself
  so that two profiles taken weeks apart at the same station stay
  apart).
- `depth_round_m` — `depth_m` rounded to `DEPTH_ROUND_DECIMALS` places.
  Used as part of the replicate-group key so casts at "essentially the
  same depth" aggregate together.

A full column inventory is generated for the audit trail.


In [ ]:
if display is not None:
    display(tables["tables/column_inventory.csv"].head(20))
else:
    print(tables["tables/column_inventory.csv"].head(20).to_string(index=False))

## Duplicate checks

A "duplicate" here is any row whose `duplicate_keys` tuple
(`[sample_id, replicate_id, sample_date, station_id, depth_m]` by
default) collides with at least one other row. Two outputs:

- `dup_rows` — every row in a duplicate group (sorted by keys).
- `dup_summary` — one row per duplicate group, with min/max/range
  for each carbonate-chemistry metric. Lets an analyst spot
  same-record disagreements at a glance.

Two new columns are added to the main frame:
`flag_duplicate` and `duplicate_group_size`.


In [ ]:
print(f"Duplicate rows  : {counts['duplicate_rows']:,}")
print(f"Duplicate groups: {counts['duplicate_groups']:,}")
print(f"Duplicate keys  : {result.manifest['duplicate_keys_used']}")
if display is not None and not tables["tables/duplicate_summary.csv"].empty:
    display(tables["tables/duplicate_summary.csv"].head(20))

## Replicate harmonisation

Groups rows by `replicate_group_keys` (default:
`[cruise_id, transect_id, station_id, depth_round_m, sample_date]`) and
computes:

- **Means** of every whitelisted numeric variable
  (`replicate_mean_vars`).
- **Sample SDs** (ddof=1) of those same variables.
- **First-value** of every other column (metadata, IDs, source
  pointers...). Conflicts in those are surfaced separately in
  `consistency_df`, not silently dropped.
- **Per-class conflict flags** (metadata / qc / provenance / source /
  other) at the row level.
- **SD-threshold flag** if any metric's group SD exceeds its
  per-metric threshold. Defaults match the GOA-ON "weather"
  precision objectives (Newton et al. 2015): pH ±0.02, TA ±10 µmol/kg.

Three tables come out of this:
- `rep_mean_sd` — one row per replicate group, fully aggregated.
- `consistency_df` — one row per (group, conflicting field).
- `disagree_df` — one row per (group, metric) where the SD exceeds
  the threshold.


In [ ]:
print(f"Replicate group keys   : {result.manifest['replicate_group_keys_used']}")
print(f"Numeric mean variables : {len(result.manifest['replicate_mean_vars_used'])}")
print(f"Replicate groups       : {counts['replicate_groups']:,}")
print(f"Consistency conflicts  : {counts['consistency_conflicts']:,}")
print(f"SD threshold failures  : {counts['sd_failures']:,}")
print(f"SD thresholds in use   : {result.manifest['replicate_sd_thresholds']}")
if display is not None:
    for rel in ("tables/replicate_consistency.csv", "tables/replicate_disagreement.csv"):
        if not tables[rel].empty:
            display(tables[rel].head(20))

## Quick preview of the enhanced table

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "station_id",
        "depth_m", "depth_round_m", "sample_month",
        "ta_best_umolkg", "ph_best", "ph_co2sys",
        "pco2_best_uatm", "dic_best_umol_kg",
        "flag_duplicate", "duplicate_group_size",
        "replicate_group_n", "flag_has_replicates",
        "flag_replicate_any_conflict", "flag_replicate_sd_exceeded",
    ]
    if c in df.columns
]

if display is not None:
    display(df[preview_cols].head(20))
else:
    print(df[preview_cols].head(20).to_string(index=False))


## Write outputs

```
<OUT_DIR>/
    data/enhanced.csv   (and .parquet)   ◄── Stage 3 input
    tables/column_inventory.csv, canonical_presence.csv, duplicate_rows.csv,
           duplicate_summary.csv, replicate_means_sd.csv,
           replicate_consistency.csv, replicate_disagreement.csv
    reports/report.md
    logs/manifest.json, effective_config.json
    run_bundle_manifest.json
```

`DRY_RUN = True` skips the writes. `NO_PARQUET = True` writes CSV only.
The run bundle hashes the input, the config and every file this run
wrote, so `roap verify-bundle <OUT_DIR>` can check them later.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    written = write_stage_result(result, out_root, parquet=not NO_PARQUET)
    print(f"Stage 2 complete: {len(written)} files under {out_root}")
    print(f"  -> Stage 3 input: {written['data/enhanced.csv']}")